<a href="https://colab.research.google.com/github/shakhaoathpappu-jpg/Datathon-2026-North-South-University/blob/main/Copy_of_Untitled02_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q duckdb lightgbm catboost scikit-learn pandas numpy

from google.colab import drive
drive.mount('/content/drive')

import duckdb, pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.0 MB/s eta 0:00:00
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
DATA_DIR = "/content/drive/MyDrive/public"

con = duckdb.connect()
con.execute("PRAGMA threads=4;")

TRX = f"{DATA_DIR}/transactions/*.parquet"
BAL = f"{DATA_DIR}/dayend_balance/*.parquet"
KYC = f"{DATA_DIR}/kyc.parquet"

train = pd.read_csv(f"{DATA_DIR}/train_labels.csv")
test  = pd.read_csv(f"{DATA_DIR}/test.csv")

label = "CHURN"

In [ ]:
ref_date = con.execute(f"""
SELECT max(CAST(TRX_DATETIME AS TIMESTAMP))
FROM read_parquet('{TRX}')
""").fetchone()[0]

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
trx = con.execute(f"""
WITH t AS (
    SELECT *,
        CAST(TRX_DATETIME AS TIMESTAMP) AS TS
    FROM read_parquet('{TRX}')
)

SELECT
    SRC_ACCOUNT AS ACCOUNT_ID,

    COUNT(*) AS total_trx,
    SUM(TRX_AMT) AS total_amt,
    AVG(TRX_AMT) AS avg_amt,
    STDDEV(TRX_AMT) AS std_amt,

    COUNT(*) FILTER (TS >= TIMESTAMP '{ref_date}' - INTERVAL 7 DAY) AS trx_7d,
    COUNT(*) FILTER (TS >= TIMESTAMP '{ref_date}' - INTERVAL 30 DAY) AS trx_30d,
    COUNT(*) FILTER (TS >= TIMESTAMP '{ref_date}' - INTERVAL 90 DAY) AS trx_90d,

    SUM(TRX_AMT) FILTER (TS >= TIMESTAMP '{ref_date}' - INTERVAL 30 DAY) AS amt_30d,

    COUNT(*) FILTER (TRX_TYPE='P2P') AS p2p_cnt,
    COUNT(*) FILTER (TRX_TYPE='CashIn') AS cashin_cnt,
    COUNT(*) FILTER (TRX_TYPE='CashOut') AS cashout_cnt,
    COUNT(*) FILTER (TRX_TYPE='MerchantPay') AS merchant_cnt,
    COUNT(*) FILTER (TRX_TYPE='BillPay') AS bill_cnt,

    COUNT(DISTINCT DST_ACCOUNT) AS unique_dst,

    DATE_DIFF('day', MAX(TS), TIMESTAMP '{ref_date}') AS recency

FROM t
GROUP BY 1
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
trx_in = con.execute(f"""
SELECT
    DST_ACCOUNT AS ACCOUNT_ID,
    COUNT(*) AS in_trx,
    SUM(TRX_AMT) AS in_amt,
    AVG(TRX_AMT) AS in_avg,
    DATE_DIFF('day', MAX(CAST(TRX_DATETIME AS TIMESTAMP)),
              TIMESTAMP '{ref_date}') AS in_recency
FROM read_parquet('{TRX}')
GROUP BY 1
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
bal = con.execute(f"""
SELECT
    ACCOUNT_ID,

    AVG(AVAILABLE_BALANCE) AS bal_mean,
    STDDEV(AVAILABLE_BALANCE) AS bal_std,
    MIN(AVAILABLE_BALANCE) AS bal_min,
    MAX(AVAILABLE_BALANCE) AS bal_max,

    LAST(AVAILABLE_BALANCE) AS bal_last,

    COUNT(*) FILTER (AVAILABLE_BALANCE <= 0) * 1.0 / COUNT(*) AS zero_ratio,

    MAX(AVAILABLE_BALANCE) - MIN(AVAILABLE_BALANCE) AS bal_range

FROM read_parquet('{BAL}')
GROUP BY 1
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
kyc = pd.read_parquet(KYC)

kyc = kyc[kyc["ACCOUNT_TYPE"]=="Customer"].copy()

kyc["account_age"] = (
    (ref_date - pd.to_datetime(kyc["ACCOUNT_OPEN_DATE"])).dt.days
)

kyc = kyc[["ACCOUNT_ID","GENDER","REGION","account_age"]]

In [ ]:
def build(df):
    df = df.merge(kyc, on="ACCOUNT_ID", how="left")
    df = df.merge(trx, on="ACCOUNT_ID", how="left")
    df = df.merge(trx_in, on="ACCOUNT_ID", how="left")
    df = df.merge(bal, on="ACCOUNT_ID", how="left")

    # fill
    df = df.fillna(0)

    # log transform (VERY IMPORTANT)
    for c in ["total_trx","total_amt","in_trx","in_amt"]:
        df[c] = np.log1p(df[c])

    # ratios (strong signal)
    df["cashout_ratio"] = df["cashout_cnt"] / (df["total_trx"] + 1)
    df["p2p_ratio"] = df["p2p_cnt"] / (df["total_trx"] + 1)

    df["activity_drop"] = df["trx_7d"] / (df["trx_30d"]/4 + 1e-6)

    # categorical
    df["GENDER"] = df["GENDER"].astype("category")
    df["REGION"] = df["REGION"].astype("category")

    return df

train_df = build(train)
test_df  = build(test)

In [ ]:
features = [c for c in train_df.columns if c not in ["ACCOUNT_ID","CHURN"]]

X_tr, X_va, y_tr, y_va = train_test_split(
    train_df[features],
    train_df["CHURN"],
    test_size=0.2,
    stratify=train_df["CHURN"],
    random_state=42
)

In [ ]:
import lightgbm as lgb

lgb_model = lgb.LGBMClassifier(
    n_estimators=5000,
    learning_rate=0.01,
    num_leaves=127,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_samples=50,
    reg_alpha=0.5,
    reg_lambda=1.0,
    scale_pos_weight=(y_tr==0).sum()/(y_tr==1).sum(),
    random_state=42
)

lgb_model.fit(
    X_tr, y_tr,
    eval_set=[(X_va,y_va)],
    eval_metric="auc",
    callbacks=[lgb.early_stopping(100)]
)

p1 = lgb_model.predict_proba(X_va)[:,1]
print("LGB AUC:", roc_auc_score(y_va,p1))

[LightGBM] [Info] Number of positive: 60348, number of negative: 415652
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.224014 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 6824
[LightGBM] [Info] Number of data points in the train set: 476000, number of used features: 32
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.126782 -> initscore=-1.929721
[LightGBM] [Info] Start training from score -1.929721
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[227]	valid_0's auc: 0.983695	valid_0's binary_logloss: 0.158024
LGB AUC: 0.983695210623839


In [ ]:
cat_cols = ["GENDER", "REGION"]

train_cat = train_df.copy()
test_cat = test_df.copy()

for c in cat_cols:
    train_cat[c] = train_cat[c].astype(str)
    test_cat[c] = test_cat[c].astype(str)

In [ ]:
lgb_model.fit(train_df[features], train_df["CHURN"])

[LightGBM] [Info] Number of positive: 75435, number of negative: 519565
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.271354 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 6828
[LightGBM] [Info] Number of data points in the train set: 595000, number of used features: 32
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.126782 -> initscore=-1.929721
[LightGBM] [Info] Start training from score -1.929721


LGBMClassifier(colsample_bytree=0.8, learning_rate=0.01, min_child_samples=50,
               n_estimators=5000, num_leaves=127, random_state=42,
               reg_alpha=0.5, reg_lambda=1.0,
               scale_pos_weight=np.float64(6.887585338370783), subsample=0.8)

In [ ]:
from catboost import CatBoostClassifier

cat_features_idx = [train_cat[features].columns.get_loc(c) for c in cat_cols]

cat_model = CatBoostClassifier(
    iterations=1200,   # ⚡ reduced for speed
    learning_rate=0.05,
    depth=8,
    loss_function="Logloss",
    eval_metric="AUC",
    verbose=200,
    random_seed=42
)

cat_model.fit(
    train_cat[features],
    train_cat["CHURN"],
    cat_features=cat_features_idx
)

0:	total: 833ms	remaining: 16m 39s
200:	total: 2m 52s	remaining: 14m 18s
400:	total: 5m 41s	remaining: 11m 20s
600:	total: 8m 30s	remaining: 8m 28s
800:	total: 11m 19s	remaining: 5m 38s
1000:	total: 14m 8s	remaining: 2m 48s
1199:	total: 16m 56s	remaining: 0us


CatBoostClassifier(depth=8, eval_metric='AUC', iterations=1200, learning_rate=0.05, loss_function='Logloss', random_seed=42, verbose=200)

In [ ]:
p1 = lgb_model.predict_proba(test_df[features])[:,1]
p2 = cat_model.predict_proba(test_cat[features])[:,1]

pred = 0.6*p1 + 0.4*p2

In [ ]:
sub = pd.DataFrame({
    "ACCOUNT_ID": test_df["ACCOUNT_ID"],
    "CHURN_PROB": pred
})

sub.to_csv("predictions.csv", index=False)
sub.head()

,ACCOUNT_ID,CHURN_PROB
0,CUST00074385,0.965193
1,CUST00290073,0.000083
2,CUST00247934,0.000978
3,CUST00314276,0.335223
4,CUST00522755,0.607776


In [ ]:
save_path = "/content/drive/MyDrive/predictions.csv"
sub.to_csv(save_path, index=False)
print("Saved to:", save_path)

Saved to: /content/drive/MyDrive/predictions.csv


In [ ]:
import os

# কোথায় save হয়েছে খুঁজে বের করো
for root, dirs, files in os.walk("/content"):
    for f in files:
        if f == "predictions.csv":
            path = os.path.join(root, f)
            print("Found:", path)

# সরাসরি download করো
from google.colab import files
files.download("/content/predictions.csv")


Found: /content/predictions.csv
Found: /content/drive/MyDrive/predictions.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
